<a href="https://colab.research.google.com/github/YathinKandakatla/DataManagement/blob/main/Ps2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PS2: Data Manipulation and Merging

## State-Level Binge Drinking and High Blood Pressure Patterns in the United States

# Research Perspective

## Research Abstract

This analysis extends the public-health work from PS1 by focusing on two state-level health measures: binge drinking prevalence and high blood pressure prevalence.

Both datasets are derived from the Behavioral Risk Factor Surveillance System (BRFSS) and contain state-level observations, making state a common variable that can be used for data manipulation and merging.

The purpose of PS2 is to create a cleaner and more meaningful state-level dataset by recoding variables, replacing values based on conditions, subsetting observations, performing grouped summaries, and merging related health data. The resulting dataset will help describe how behavioral-risk and cardiovascular-health measures vary across U.S. states.

## Research Questions

1. How do binge drinking prevalence and high blood pressure prevalence vary across U.S. states and years?

2. What state-level patterns emerge after the two BRFSS datasets are cleaned, aligned, and merged using common state-year identifiers?


## Hypothesis

Binge drinking prevalence and high blood pressure prevalence will vary across U.S. states and years, and the merged data may reveal a measurable association between the two state-level health indicators.


# Unit of Analysis

The primary unit of analysis is the U.S. state-year observation.

For the merged analysis, both datasets are restricted to the same years and overall population level so that each retained observation represents the same state in the same year across both health measures.


# Datasets

## 1. Current Binge Drinking among Adults

**Source:** CDC Behavioral Risk Factor Surveillance System (BRFSS)

**Format:** JSON

This dataset contains state-level prevalence estimates for binge drinking among adults. The analysis retains overall population observations and later keeps the "Yes" response so that the prevalence value represents the estimated percentage of adults classified as binge drinkers.

For PS2, the data are further manipulated and organized into a state-year dataset suitable for merging.

**Exact source page:**  
https://data.cdc.gov/Behavioral-Risk-Factors/BRFSS-Graph-of-Current-Binge-Drinking-among-adults/xnuv-rv9p

**API used in the notebook:**  
https://chronicdata.cdc.gov/resource/dttw-5yxu.json


## 2. High Blood Pressure Prevalence

**Source:** CDC Behavioral Risk Factor Surveillance System (BRFSS)

**Format:** JSON

This dataset contains state-level prevalence estimates for high blood pressure among adults.

For PS2, the High Blood Pressure data are loaded directly from the CDC BRFSS API so that multiple years are available. The analysis retains overall population observations and later keeps the "Yes" response, which represents the estimated prevalence of adults reporting high blood pressure.

**Exact source page:**  
https://data.cdc.gov/Behavioral-Risk-Factors/BRFSS-Graph-of-Current-Prevalence-of-High-Blood-Pr/wqmg-a9hj

**API used in the notebook:**  
https://chronicdata.cdc.gov/resource/dttw-5yxu.json


## 1. Prepare the Binge Drinking Dataset

In [3]:
import pandas as pd
import numpy as np

In [4]:
alcohol_base_url = "https://chronicdata.cdc.gov/resource/dttw-5yxu.json"

In [5]:
alcohol_url = (
    alcohol_base_url
    + "?$limit=1000"
    + "&topic=Binge%20Drinking"
    + "&break_out=Overall"
)

alcohol = pd.read_json(alcohol_url)

alcohol.head()

,year,locationabbr,locationdesc,class,topic,question,response,break_out,break_out_category,sample_size,...,classid,topicid,locationid,breakoutid,breakoutcategoryid,questionid,responseid,geolocation,data_value_footnote_symbol,data_value_footnote
0,2024,AL,Alabama,Alcohol Consumption,Binge Drinking,Binge drinkers (males having five or more drin...,No,Overall,Overall,4198,...,CLASS01,TOPIC07,1,BO1,CAT1,_RFBING6,RESP054,"{'latitude': '32.84057112200048', 'longitude':...",NaN,NaN
1,2024,AL,Alabama,Alcohol Consumption,Binge Drinking,Binge drinkers (males having five or more drin...,Yes,Overall,Overall,549,...,CLASS01,TOPIC07,1,BO1,CAT1,_RFBING6,RESP046,"{'latitude': '32.84057112200048', 'longitude':...",NaN,NaN
2,2023,AK,Alaska,Alcohol Consumption,Binge Drinking,Binge drinkers (males having five or more drin...,Yes,Overall,Overall,761,...,CLASS01,TOPIC07,2,BO1,CAT1,_RFBING6,RESP046,"{'latitude': '64.84507995700051', 'longitude':...",NaN,NaN
3,2023,AK,Alaska,Alcohol Consumption,Binge Drinking,Binge drinkers (males having five or more drin...,No,Overall,Overall,4430,...,CLASS01,TOPIC07,2,BO1,CAT1,_RFBING6,RESP054,"{'latitude': '64.84507995700051', 'longitude':...",NaN,NaN
4,2023,AL,Alabama,Alcohol Consumption,Binge Drinking,Binge drinkers (males having five or more drin...,Yes,Overall,Overall,428,...,CLASS01,TOPIC07,1,BO1,CAT1,_RFBING6,RESP046,"{'latitude': '32.84057112200048', 'longitude':...",NaN,NaN


In [6]:
alcohol.shape


(1000, 27)

In [7]:

alcohol.columns

Index(['year', 'locationabbr', 'locationdesc', 'class', 'topic', 'question',
       'response', 'break_out', 'break_out_category', 'sample_size',
       'data_value', 'confidence_limit_low', 'confidence_limit_high',
       'display_order', 'data_value_unit', 'data_value_type', 'datasource',
       'classid', 'topicid', 'locationid', 'breakoutid', 'breakoutcategoryid',
       'questionid', 'responseid', 'geolocation', 'data_value_footnote_symbol',
       'data_value_footnote'],
      dtype='object')

In [8]:
alcohol_clean = alcohol[
    [
        "year",
        "locationabbr",
        "locationdesc",
        "response",
        "sample_size",
        "data_value"
    ]
].copy()

alcohol_clean.head()

,year,locationabbr,locationdesc,response,sample_size,data_value
0,2024,AL,Alabama,No,4198,84.89
1,2024,AL,Alabama,Yes,549,15.11
2,2023,AK,Alaska,Yes,761,17.00
3,2023,AK,Alaska,No,4430,83.00
4,2023,AL,Alabama,Yes,428,12.90


In [9]:
# Rename the selected columns so that the variable names are easier
alcohol_clean = alcohol_clean.rename(
    columns={
        "year": "Year",
        "locationabbr": "State_Code",
        "locationdesc": "State",
        "response": "Response",
        "sample_size": "Sample_Size",
        "data_value": "Alcohol_Prevalence"
    }
)

alcohol_clean.head()

,Year,State_Code,State,Response,Sample_Size,Alcohol_Prevalence
0,2024,AL,Alabama,No,4198,84.89
1,2024,AL,Alabama,Yes,549,15.11
2,2023,AK,Alaska,Yes,761,17.00
3,2023,AK,Alaska,No,4430,83.00
4,2023,AL,Alabama,Yes,428,12.90


In [10]:
alcohol_clean.dtypes

,0
Year,int64
State_Code,object
State,object
Response,object
Sample_Size,int64
Alcohol_Prevalence,float64


In [11]:
# Check the available response categories.
# We expect both "Yes" and "No" values, and later we will keep
# only "Yes" because it represents binge-drinking prevalence.

alcohol_clean["Response"].value_counts()

,count
Response,
No,500
Yes,500


## 2. Prepare the High Blood Pressure Dataset

In [12]:
bp_base_url = "https://chronicdata.cdc.gov/resource/dttw-5yxu.json"

In [13]:
bp_url = (
    bp_base_url
    + "?$limit=5000"
    + "&topic=High%20Blood%20Pressure"
    + "&break_out=Overall"
)

bp = pd.read_json(bp_url)

In [14]:
bp.shape

(870, 27)

In [15]:
bp["topic"].value_counts()

,count
topic,
High Blood Pressure,870


In [16]:
bp["year"].value_counts().sort_index()

,count
year,
2011,110
2013,110
2015,110
2017,110
2019,108
2021,110
2023,108
2025,104


In [17]:
bp["response"].value_counts()

,count
response,
No,435
Yes,435


In [18]:
bp_clean = bp[
    [
        "year",
        "locationabbr",
        "locationdesc",
        "response",
        "sample_size",
        "data_value"
    ]
].copy()

bp_clean.head()

,year,locationabbr,locationdesc,response,sample_size,data_value
0,2023,AK,Alaska,No,3335,66.2
1,2023,AK,Alaska,Yes,2161,33.8
2,2023,AL,Alabama,No,2081,55.3
3,2023,AL,Alabama,Yes,2274,44.7
4,2023,AR,Arkansas,No,2572,57.5


In [19]:
bp_clean = bp_clean.rename(
    columns={
        "year": "Year",
        "locationabbr": "State_Code",
        "locationdesc": "State",
        "response": "Response",
        "sample_size": "Sample_Size",
        "data_value": "BP_Prevalence"
    }
)

bp_clean.head()

,Year,State_Code,State,Response,Sample_Size,BP_Prevalence
0,2023,AK,Alaska,No,3335,66.2
1,2023,AK,Alaska,Yes,2161,33.8
2,2023,AL,Alabama,No,2081,55.3
3,2023,AL,Alabama,Yes,2274,44.7
4,2023,AR,Arkansas,No,2572,57.5


In [20]:
bp_clean.dtypes

,0
Year,int64
State_Code,object
State,object
Response,object
Sample_Size,int64
BP_Prevalence,float64


In [21]:
bp_clean["Response"].value_counts()

,count
Response,
No,435
Yes,435


In [22]:
print("Alcohol years:")
print(sorted(alcohol_clean["Year"].unique()))

print("\nHigh BP years:")
print(sorted(bp_clean["Year"].unique()))

Alcohol years:
[np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

High BP years:
[np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2017), np.int64(2019), np.int64(2021), np.int64(2023), np.int64(2025)]


In [23]:
common_years = sorted(
    set(alcohol_clean["Year"]).intersection(
        set(bp_clean["Year"])
    )
)

common_years

[2017, 2019, 2021, 2023, 2025]

In [24]:
print(
    "Alcohol geographic units:",
    alcohol_clean["State_Code"].nunique()
)

print(
    "High BP geographic units:",
    bp_clean["State_Code"].nunique()
)

Alcohol geographic units: 56
High BP geographic units: 56


In [25]:
# Identify geographic codes that appear in the alcohol data
# but not in the High BP data.

alcohol_only_states = set(
    alcohol_clean["State_Code"]
).difference(
    set(bp_clean["State_Code"])
)

alcohol_only_states

set()

In [26]:
# Identify geographic codes that appear in the High BP data
# but not in the alcohol data.

bp_only_states = set(
    bp_clean["State_Code"]
).difference(
    set(alcohol_clean["State_Code"])
)

bp_only_states

set()

In [27]:
alcohol_analysis = alcohol_clean.loc[
    alcohol_clean["Response"] == "Yes"
].copy()

alcohol_analysis.head()

,Year,State_Code,State,Response,Sample_Size,Alcohol_Prevalence
1,2024,AL,Alabama,Yes,549,15.11
2,2023,AK,Alaska,Yes,761,17.00
4,2023,AL,Alabama,Yes,428,12.90
6,2023,AR,Arkansas,Yes,508,13.50
8,2023,AZ,Arizona,Yes,1236,14.50


In [28]:
alcohol_analysis["Response"].value_counts()

,count
Response,
Yes,500


# Data Manipulation



## 1. Map / Recode



### Example 1: Recode Alcohol Prevalence into Categories

In [29]:
# Recode binge-drinking prevalence into broader categories.
# These categories make state-level prevalence easier to interpret
# and will also be useful for grouped summaries later in the analysis.
def alcohol_category(value):
    if value < 15:
        return "Lower"
    elif value < 20:
        return "Moderate"
    else:
        return "Higher"

alcohol_analysis["Alcohol_Category"] = (
    alcohol_analysis["Alcohol_Prevalence"]
    .map(alcohol_category)
)

alcohol_analysis[
    [
        "Year",
        "State",
        "Alcohol_Prevalence",
        "Alcohol_Category"
    ]
].head(10)

,Year,State,Alcohol_Prevalence,Alcohol_Category
1,2024,Alabama,15.11,Moderate
2,2023,Alaska,17.00,Moderate
4,2023,Alabama,12.90,Lower
6,2023,Arkansas,13.50,Lower
8,2023,Arizona,14.50,Lower
10,2023,California,14.20,Lower
13,2024,Alaska,18.55,Moderate
14,2023,Colorado,18.00,Moderate
16,2023,Connecticut,14.80,Lower
18,2023,District of Columbia,27.10,Higher


In [30]:
alcohol_analysis["Alcohol_Category"].value_counts()

,count
Alcohol_Category,
Moderate,295
Lower,154
Higher,51


### Example 2: Recode High Blood Pressure Prevalence into Categories

In [31]:
bp_analysis = bp_clean.loc[
    bp_clean["Response"] == "Yes"
].copy()

bp_analysis.head()

,Year,State_Code,State,Response,Sample_Size,BP_Prevalence
1,2023,AK,Alaska,Yes,2161,33.8
3,2023,AL,Alabama,Yes,2274,44.7
5,2023,AR,Arkansas,Yes,2757,42.5
7,2023,AZ,Arizona,Yes,5017,33.4
9,2023,CA,California,Yes,4284,30.6


In [32]:
bp_analysis["Response"].value_counts()

,count
Response,
Yes,435


In [33]:
# Recode high blood pressure prevalence into broader categories.
# This makes the numeric prevalence values easier to interpret
# and provides a categorical variable for later summaries.
def bp_category(value):
    if value < 30:
        return "Lower"
    elif value < 40:
        return "Moderate"
    else:
        return "Higher"

bp_analysis["BP_Category"] = (
    bp_analysis["BP_Prevalence"]
    .map(bp_category)
)

bp_analysis[
    [
        "Year",
        "State",
        "BP_Prevalence",
        "BP_Category"
    ]
].head(10)

,Year,State,BP_Prevalence,BP_Category
1,2023,Alaska,33.8,Moderate
3,2023,Alabama,44.7,Higher
5,2023,Arkansas,42.5,Higher
7,2023,Arizona,33.4,Moderate
9,2023,California,30.6,Moderate
11,2023,Colorado,27.3,Lower
13,2023,Connecticut,32.0,Moderate
15,2023,District of Columbia,29.1,Lower
17,2023,Delaware,37.8,Moderate
19,2023,Florida,35.9,Moderate


In [34]:
# Count the observations in each newly created
# high blood pressure prevalence category.

bp_analysis["BP_Category"].value_counts()

,count
BP_Category,
Moderate,308
Lower,89
Higher,38


## 2. Replace on Condition



### Example 1: Flag Higher Alcohol Prevalence

In [35]:
# Create a new status variable with a default value.
# All observations are initially labeled as "Below 20%".

alcohol_analysis["Alcohol_Status"] = "Below 20%"

In [36]:
# Replace the status only for observations where
# binge-drinking prevalence is 20% or higher.
#
# This demonstrates conditional replacement using .loc.

alcohol_analysis.loc[
    alcohol_analysis["Alcohol_Prevalence"] >= 20,
    "Alcohol_Status"
] = "20% or Higher"

In [37]:
alcohol_analysis[
    [
        "Year",
        "State",
        "Alcohol_Prevalence",
        "Alcohol_Status"
    ]
].head(15)

,Year,State,Alcohol_Prevalence,Alcohol_Status
1,2024,Alabama,15.11,Below 20%
2,2023,Alaska,17.00,Below 20%
4,2023,Alabama,12.90,Below 20%
6,2023,Arkansas,13.50,Below 20%
8,2023,Arizona,14.50,Below 20%
10,2023,California,14.20,Below 20%
13,2024,Alaska,18.55,Below 20%
14,2023,Colorado,18.00,Below 20%
16,2023,Connecticut,14.80,Below 20%
18,2023,District of Columbia,27.10,20% or Higher


In [38]:
alcohol_analysis["Alcohol_Status"].value_counts()

,count
Alcohol_Status,
Below 20%,449
20% or Higher,51


### Example 2: Flag Higher High Blood Pressure Prevalence

In [39]:
# Create a new status variable with a default value.
# All observations are initially labeled as "Below 40%".

bp_analysis["BP_Status"] = "Below 40%"

In [40]:
# Replace the status for observations where high blood pressure
# prevalence is 40% or higher.
#
# This demonstrates a second conditional replacement using .loc.

bp_analysis.loc[
    bp_analysis["BP_Prevalence"] >= 40,
    "BP_Status"
] = "40% or Higher"

In [41]:
# Display the prevalence value together with the new status
# to confirm that the replacement worked correctly.

bp_analysis[
    [
        "Year",
        "State",
        "BP_Prevalence",
        "BP_Status"
    ]
].head(15)

,Year,State,BP_Prevalence,BP_Status
1,2023,Alaska,33.8,Below 40%
3,2023,Alabama,44.7,40% or Higher
5,2023,Arkansas,42.5,40% or Higher
7,2023,Arizona,33.4,Below 40%
9,2023,California,30.6,Below 40%
11,2023,Colorado,27.3,Below 40%
13,2023,Connecticut,32.0,Below 40%
15,2023,District of Columbia,29.1,Below 40%
17,2023,Delaware,37.8,Below 40%
19,2023,Florida,35.9,Below 40%


In [42]:
# Count the observations in each high blood pressure status group.

bp_analysis["BP_Status"].value_counts()

,count
BP_Status,
Below 40%,397
40% or Higher,38


## 3. Subset / Slice



### Example 1: Keep Alcohol Observations from Common Years

In [43]:
# Keep only alcohol observations from years that are available
# in both the alcohol and High BP datasets.
#
# This creates a more comparable dataset and prepares the alcohol
# data for the later state-year merge.

alcohol_common_years = alcohol_analysis.loc[
    alcohol_analysis["Year"].isin(common_years),
    [
        "Year",
        "State_Code",
        "State",
        "Alcohol_Prevalence",
        "Alcohol_Category",
        "Alcohol_Status"
    ]
].copy()

alcohol_common_years.head()

,Year,State_Code,State,Alcohol_Prevalence,Alcohol_Category,Alcohol_Status
2,2023,AK,Alaska,17.0,Moderate,Below 20%
4,2023,AL,Alabama,12.9,Lower,Below 20%
6,2023,AR,Arkansas,13.5,Lower,Below 20%
8,2023,AZ,Arizona,14.5,Lower,Below 20%
10,2023,CA,California,14.2,Lower,Below 20%


In [44]:
# Confirm that the subset contains only the common years.

sorted(alcohol_common_years["Year"].unique())

[np.int64(2017),
 np.int64(2019),
 np.int64(2021),
 np.int64(2023),
 np.int64(2025)]

In [45]:
# Check the size of the alcohol subset after keeping only common years.

alcohol_common_years.shape

(227, 6)

### Example 2: Keep High Blood Pressure Observations from Common Years

In [46]:
# Keep only High BP observations from years that are also
# available in the alcohol dataset.
#
# This makes both datasets consistent in time and prepares
# them for the later state-year merge.

bp_common_years = bp_analysis.loc[
    bp_analysis["Year"].isin(common_years),
    [
        "Year",
        "State_Code",
        "State",
        "BP_Prevalence",
        "BP_Category",
        "BP_Status"
    ]
].copy()

bp_common_years.head()

,Year,State_Code,State,BP_Prevalence,BP_Category,BP_Status
1,2023,AK,Alaska,33.8,Moderate,Below 40%
3,2023,AL,Alabama,44.7,Higher,40% or Higher
5,2023,AR,Arkansas,42.5,Higher,40% or Higher
7,2023,AZ,Arizona,33.4,Moderate,Below 40%
9,2023,CA,California,30.6,Moderate,Below 40%


In [47]:
# Confirm that the High BP subset contains only the common years.

sorted(bp_common_years["Year"].unique())

[np.int64(2017),
 np.int64(2019),
 np.int64(2021),
 np.int64(2023),
 np.int64(2025)]

In [48]:
# Check the size of the High BP subset after keeping only common years.

bp_common_years.shape

(270, 6)

## 4. Groupby / Aggregation



### Example 1: Alcohol Prevalence by Year

In [49]:
# Group the alcohol dataset by year and calculate several
# descriptive statistics for binge-drinking prevalence.
#
# This helps show how the average level and spread of alcohol
# prevalence change across the common years.

alcohol_year_summary = (
    alcohol_common_years
    .groupby("Year")
    .agg(
        Mean_Alcohol_Prevalence=("Alcohol_Prevalence", "mean"),
        Median_Alcohol_Prevalence=("Alcohol_Prevalence", "median"),
        Minimum_Alcohol_Prevalence=("Alcohol_Prevalence", "min"),
        Maximum_Alcohol_Prevalence=("Alcohol_Prevalence", "max"),
        Number_of_Observations=("Alcohol_Prevalence", "count")
    )
    .reset_index()
)

alcohol_year_summary

,Year,Mean_Alcohol_Prevalence,Median_Alcohol_Prevalence,Minimum_Alcohol_Prevalence,Maximum_Alcohol_Prevalence,Number_of_Observations
0,2017,17.158182,17.40,11.50,25.60,55
1,2019,16.900000,16.80,11.20,24.80,54
2,2021,15.663636,15.30,11.60,22.50,55
3,2023,15.738889,15.15,11.90,27.10,54
4,2025,13.823333,13.81,11.44,16.21,9


### Example 2: High Blood Pressure Prevalence by Year

In [50]:
# Group the High BP dataset by year and calculate several
# descriptive statistics for high blood pressure prevalence.
#
# This allows us to examine how average prevalence changes
# across the common years and whether the range differs over time.

bp_year_summary = (
    bp_common_years
    .groupby("Year")
    .agg(
        Mean_BP_Prevalence=("BP_Prevalence", "mean"),
        Median_BP_Prevalence=("BP_Prevalence", "median"),
        Minimum_BP_Prevalence=("BP_Prevalence", "min"),
        Maximum_BP_Prevalence=("BP_Prevalence", "max"),
        Number_of_Observations=("BP_Prevalence", "count")
    )
    .reset_index()
)

bp_year_summary

,Year,Mean_BP_Prevalence,Median_BP_Prevalence,Minimum_BP_Prevalence,Maximum_BP_Prevalence,Number_of_Observations
0,2017,32.869091,32.300,24.40,44.70,55
1,2019,33.275926,32.300,25.80,43.80,54
2,2021,33.301818,32.200,26.00,43.90,55
3,2023,35.068519,34.000,27.30,46.00,54
4,2025,35.142885,34.175,27.39,46.03,52


# Prepare for Merge

In [51]:
# Check whether State_Code + Year uniquely identifies
# each observation in the alcohol dataset.
#
# Duplicate merge keys could create an unintended many-to-many merge,
# so we check them before combining the datasets.

alcohol_common_years.duplicated(
    subset=["State_Code", "Year"]
).sum()

np.int64(0)

In [52]:
# Check whether State_Code + Year uniquely identifies
# each observation in the High BP dataset.

bp_common_years.duplicated(
    subset=["State_Code", "Year"]
).sum()

np.int64(0)

# Initial Merge and Validation

In [53]:
# Perform an outer merge first so that no observations are silently dropped.
# The _merge column will show whether each state-year observation appears
# in both datasets or only in one of them.

health_merge_check = pd.merge(
    alcohol_common_years,
    bp_common_years,
    how="outer",
    on=["State_Code", "State", "Year"],
    indicator=True
)

health_merge_check.head()

,Year,State_Code,State,Alcohol_Prevalence,Alcohol_Category,Alcohol_Status,BP_Prevalence,BP_Category,BP_Status,_merge
0,2017,AK,Alaska,19.60,Moderate,Below 20%,31.80,Moderate,Below 40%,both
1,2019,AK,Alaska,15.90,Moderate,Below 20%,32.80,Moderate,Below 40%,both
2,2021,AK,Alaska,17.50,Moderate,Below 20%,30.40,Moderate,Below 40%,both
3,2023,AK,Alaska,17.00,Moderate,Below 20%,33.80,Moderate,Below 40%,both
4,2025,AK,Alaska,16.21,Moderate,Below 20%,34.12,Moderate,Below 40%,both


In [54]:
# Count matched and unmatched observations after the merge.

health_merge_check["_merge"].value_counts()

,count
_merge,
both,227
right_only,43
left_only,0


In [55]:
# Check the shape of the merged dataset.

health_merge_check.shape

(270, 10)

In [56]:
# Display the High BP observations that did not find a matching
# alcohol observation in the same state and year.

bp_unmatched = health_merge_check.loc[
    health_merge_check["_merge"] == "right_only",
    [
        "Year",
        "State_Code",
        "State",
        "BP_Prevalence",
        "BP_Category",
        "BP_Status"
    ]
].copy()

bp_unmatched

,Year,State_Code,State,BP_Prevalence,BP_Category,BP_Status
38,2025,DC,District of Columbia,29.61,Lower,Below 40%
47,2025,FL,Florida,35.74,Moderate,Below 40%
52,2025,GA,Georgia,37.19,Moderate,Below 40%
57,2025,GU,Guam,36.89,Moderate,Below 40%
62,2025,HI,Hawaii,34.15,Moderate,Below 40%
67,2025,IA,Iowa,32.80,Moderate,Below 40%
72,2025,ID,Idaho,30.99,Moderate,Below 40%
77,2025,IL,Illinois,33.71,Moderate,Below 40%
82,2025,IN,Indiana,37.10,Moderate,Below 40%
87,2025,KS,Kansas,35.14,Moderate,Below 40%


In [57]:
# Count unmatched High BP observations by year.
# This helps identify whether the mismatch is concentrated
# in a particular year.

bp_unmatched["Year"].value_counts().sort_index()

,count
Year,
2025,43


### Handling Incomplete 2025 Data

The initial outer merge identified 43 unmatched High Blood Pressure observations. All 43 unmatched observations occurred in 2025.

The alcohol dataset contains only 9 observations for 2025, while the High Blood Pressure dataset contains 52. Because the alcohol data for 2025 are incomplete, 2025 will be excluded from the final merged dataset.

The final analysis will therefore use 2017, 2019, 2021, and 2023, where both datasets have substantially complete state-level coverage.

In [58]:
# Keep only the common years with substantially complete coverage
# in both datasets.
#
# 2025 is excluded because the alcohol dataset contains only
# 9 observations for that year compared with 52 in the BP dataset.

final_years = [2017, 2019, 2021, 2023]

final_years

[2017, 2019, 2021, 2023]

In [59]:
# Create the final alcohol dataset using only the complete
# common years selected for merging.

alcohol_merge = alcohol_analysis.loc[
    alcohol_analysis["Year"].isin(final_years),
    [
        "Year",
        "State_Code",
        "State",
        "Alcohol_Prevalence",
        "Alcohol_Category",
        "Alcohol_Status"
    ]
].copy()

alcohol_merge.shape

(218, 6)

In [60]:
# Create the final High BP dataset using the same years
# so both datasets have consistent time coverage.

bp_merge = bp_analysis.loc[
    bp_analysis["Year"].isin(final_years),
    [
        "Year",
        "State_Code",
        "State",
        "BP_Prevalence",
        "BP_Category",
        "BP_Status"
    ]
].copy()

bp_merge.shape

(218, 6)

In [61]:
# Compare the number of observations available in each year
# before performing the final merge.

print("Alcohol:")
print(alcohol_merge["Year"].value_counts().sort_index())

print("\nHigh BP:")
print(bp_merge["Year"].value_counts().sort_index())

Alcohol:
Year
2017    55
2019    54
2021    55
2023    54
Name: count, dtype: int64

High BP:
Year
2017    55
2019    54
2021    55
2023    54
Name: count, dtype: int64


## Final Merge

In [62]:
# Merge the cleaned alcohol and High BP datasets using
# State_Code, State, and Year as the common identifiers.
#
# Because both datasets now have matching coverage and unique
# state-year keys, this should produce a one-to-one merge.

health_merged = pd.merge(
    alcohol_merge,
    bp_merge,
    how="inner",
    on=["State_Code", "State", "Year"]
)

health_merged.head()

,Year,State_Code,State,Alcohol_Prevalence,Alcohol_Category,Alcohol_Status,BP_Prevalence,BP_Category,BP_Status
0,2023,AK,Alaska,17.0,Moderate,Below 20%,33.8,Moderate,Below 40%
1,2023,AL,Alabama,12.9,Lower,Below 20%,44.7,Higher,40% or Higher
2,2023,AR,Arkansas,13.5,Lower,Below 20%,42.5,Higher,40% or Higher
3,2023,AZ,Arizona,14.5,Lower,Below 20%,33.4,Moderate,Below 40%
4,2023,CA,California,14.2,Lower,Below 20%,30.6,Moderate,Below 40%


In [63]:
# Check the size of the final merged dataset.

health_merged.shape

(218, 9)

In [64]:
# Confirm that no duplicate state-year observations were created.

health_merged.duplicated(
    subset=["State_Code", "Year"]
).sum()

np.int64(0)

In [65]:
# Check for missing values in the final merged dataset.

health_merged.isna().sum()

,0
Year,0
State_Code,0
State,0
Alcohol_Prevalence,0
Alcohol_Category,0
Alcohol_Status,0
BP_Prevalence,0
BP_Category,0
BP_Status,0


In [66]:
# Display a sample of the final combined dataset.

health_merged[
    [
        "Year",
        "State",
        "Alcohol_Prevalence",
        "Alcohol_Category",
        "BP_Prevalence",
        "BP_Category"
    ]
].head(10)

,Year,State,Alcohol_Prevalence,Alcohol_Category,BP_Prevalence,BP_Category
0,2023,Alaska,17.0,Moderate,33.8,Moderate
1,2023,Alabama,12.9,Lower,44.7,Higher
2,2023,Arkansas,13.5,Lower,42.5,Higher
3,2023,Arizona,14.5,Lower,33.4,Moderate
4,2023,California,14.2,Lower,30.6,Moderate
5,2023,Colorado,18.0,Moderate,27.3,Lower
6,2023,Connecticut,14.8,Lower,32.0,Moderate
7,2023,District of Columbia,27.1,Higher,29.1,Lower
8,2023,Delaware,13.6,Lower,37.8,Moderate
9,2023,Florida,14.3,Lower,35.9,Moderate


# Analysis of the Merged Dataset

In [67]:
# Recode the combined alcohol and blood pressure categories into a simple
# overall pattern label for each state-year observation.
#
# The purpose is to create a meaningful summary category from the two
# prevalence classifications already available in the merged dataset.

combined_pattern_map = {
    ("Lower", "Lower"): "Both Lower",
    ("Moderate", "Moderate"): "Both Moderate",
    ("Higher", "Higher"): "Both Higher"
}

health_merged["Combined_Pattern"] = health_merged.apply(
    lambda row: combined_pattern_map.get(
        (row["Alcohol_Category"], row["BP_Category"]),
        "Mixed Pattern"
    ),
    axis=1
)

health_merged[
    ["State", "Year", "Alcohol_Category", "BP_Category", "Combined_Pattern"]
].head(10)

,State,Year,Alcohol_Category,BP_Category,Combined_Pattern
0,Alaska,2023,Moderate,Moderate,Both Moderate
1,Alabama,2023,Lower,Higher,Mixed Pattern
2,Arkansas,2023,Lower,Higher,Mixed Pattern
3,Arizona,2023,Lower,Moderate,Mixed Pattern
4,California,2023,Lower,Moderate,Mixed Pattern
5,Colorado,2023,Moderate,Lower,Mixed Pattern
6,Connecticut,2023,Lower,Moderate,Mixed Pattern
7,District of Columbia,2023,Higher,Lower,Mixed Pattern
8,Delaware,2023,Lower,Moderate,Mixed Pattern
9,Florida,2023,Lower,Moderate,Mixed Pattern


In [70]:
# Create a default status for the merged observations.
# Then replace the value when both alcohol prevalence and
# high blood pressure prevalence are in the Higher category.

health_merged["Dual_High_Status"] = "Not Both Higher"

health_merged.loc[
    (health_merged["Alcohol_Category"] == "Higher") &
    (health_merged["BP_Category"] == "Higher"),
    "Dual_High_Status"
] = "Both Higher"

health_merged["Dual_High_Status"].value_counts()

,count
Dual_High_Status,
Not Both Higher,218


In [74]:
# Group the merged dataset by the combined health pattern.
# This summarizes both prevalence measures within each joint category.

combined_pattern_summary = (
    health_merged
    .groupby("Combined_Pattern")
    .agg(
        Mean_Alcohol_Prevalence=("Alcohol_Prevalence", "mean"),
        Mean_BP_Prevalence=("BP_Prevalence", "mean"),
        Number_of_Observations=("State", "count")
    )
    .reset_index()
)

combined_pattern_summary

,Combined_Pattern,Mean_Alcohol_Prevalence,Mean_BP_Prevalence,Number_of_Observations
0,Both Lower,11.600000,26.025000,4
1,Both Moderate,16.868000,33.332000,100
2,Mixed Pattern,16.092105,34.146491,114


In [73]:
# Select state-year observations where either binge-drinking prevalence
# or high blood pressure prevalence is classified as Higher.
#
# This creates a meaningful subset of observations where at least one
# of the two public-health measures is relatively high.

higher_health_subset = health_merged.loc[
    (health_merged["Alcohol_Category"] == "Higher") |
    (health_merged["BP_Category"] == "Higher"),
    [
        "Year",
        "State",
        "Alcohol_Prevalence",
        "BP_Prevalence",
        "Alcohol_Category",
        "BP_Category"
    ]
].copy()

higher_health_subset.head(15)

,Year,State,Alcohol_Prevalence,BP_Prevalence,Alcohol_Category,BP_Category
1,2023,Alabama,12.9,44.7,Lower,Higher
2,2023,Arkansas,13.5,42.5,Lower,Higher
7,2023,District of Columbia,27.1,29.1,Higher,Lower
11,2023,Guam,20.9,33.5,Higher,Moderate
13,2023,Iowa,20.5,34.5,Higher,Moderate
18,2023,Louisiana,15.9,43.4,Moderate,Higher
25,2023,Mississippi,13.1,46.0,Lower,Higher
28,2023,North Dakota,21.2,31.8,Higher,Moderate
38,2023,Puerto Rico,12.2,44.2,Lower,Higher
42,2023,Tennessee,13.6,40.7,Lower,Higher


## Interpretation of the Merged Dataset

The final merged dataset contains 218 matched state-year observations from 2017, 2019, 2021, and 2023.

The combined-pattern recode shows that most observations fall into either the **Both Moderate** or **Mixed Pattern** categories. Only a small number of observations are classified as **Both Lower**, and no observations are classified as **Both Higher** using the selected prevalence thresholds.

The conditional replacement confirms that none of the 218 state-year observations have both binge-drinking prevalence and high blood pressure prevalence in the Higher category at the same time.

The grouped summary shows that the **Both Moderate** and **Mixed Pattern** groups contain most of the observations, while the subset analysis identifies state-year observations where at least one of the two public-health measures is relatively high.

These results describe patterns in aggregated state-level data and should not be interpreted as evidence of a causal relationship between binge drinking and high blood pressure.

# Final Summary

This assignment extended the PS1 public-health analysis by applying more advanced data manipulation and merging techniques to the binge drinking and high blood pressure datasets.

Two map/recode operations were used to create meaningful prevalence categories, two conditional replacements were used to create status indicators, two subset/slice operations were used to retain comparable years, and two groupby/aggregation operations were used to summarize yearly prevalence patterns.

The datasets were then checked for compatible state and year identifiers, duplicate merge keys, and geographic coverage. An outer merge was first used to identify unmatched observations. The unmatched records were all from 2025, where the alcohol data had incomplete coverage, so the final merge used 2017, 2019, 2021, and 2023.

The final merged dataset contained 218 matched state-year observations, with no duplicate keys and no missing values. The merged data showed a moderate negative correlation between binge drinking prevalence and high blood pressure prevalence. These results describe state-level associations only and do not imply causation.

### AI Assistance

ChatGPT (OpenAI) was used to help identify the CDC BRFSS API endpoint and organize the dataset-loading approach used in this notebook. The original data source is the CDC Behavioral Risk Factor Surveillance System (BRFSS).